In [118]:
import datetime as dt
import numpy as np
import pandas as pd
import numpy as np
import tabulate as tb
from typing import Dict
import tensorflow as tf
import re
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


SEED = 42
np.random.seed(SEED)
CURRENT_DATE = dt.datetime(2025, 7, 1)

In [119]:
data_players = pd.read_csv('./../data/players_data.csv')
data_players['Surface'] = data_players['Surface'].astype('category').cat.codes

dic_players = {}
for index, row in data_players.iterrows():
    dic_players[f'{row['Player']}_{row['Surface']}_{row['Period']}'] = {
        'Elo': row['Elo'],
        'Matches': row['Matches'],
        'WinRate': row['WinRate']
    }

data_matches = pd.read_csv('./../data/wta.csv')
data_matches['Date'] = pd.to_datetime(data_matches['Date'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
data_matches['Date'].dropna(inplace=True)
data_matches['Surface'] = data_matches['Surface'].astype('category').cat.codes
data_matches['WinnerBinary'] = (data_matches['Winner'] == data_matches['Player_1']).astype(int)

print(tb.tabulate(data_players.sort_values(by='Elo', ascending=True).head(), headers='keys', tablefmt='psql'))
print(tb.tabulate(data_matches.head(), headers='keys', tablefmt='psql'))

+-------+----------------+-----------+-----------+-----------+--------+----------+-----------+---------+
|       | Player         |   Surface | Period    |   Matches |   Wins |   Losses |   WinRate |     Elo |
|-------+----------------+-----------+-----------+-----------+--------+----------+-----------+---------|
| 10800 | Uchijima M.    |         4 | all       |        39 |     12 |       27 |  0.692308 | 1297.87 |
| 10801 | Uchijima M.    |         4 | last_year |        26 |      8 |       18 |  0.692308 | 1306.35 |
| 11835 | Giorgi C.      |         4 | all       |       219 |    110 |      109 |  0.497717 | 1346.45 |
|  9225 | Cocciaretto E. |         4 | all       |        63 |     23 |       40 |  0.634921 | 1347.97 |
|  1020 | Halep S.       |         4 | all       |       367 |    254 |      113 |  0.307902 | 1361.27 |
+-------+----------------+-----------+-----------+-----------+--------+----------+-----------+---------+
+----+--------------+---------------------+---------+--

C:\Users\BPX Michał Dziedziak\AppData\Local\Temp\ipykernel_17800\4171698402.py:12: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  data_matches = pd.read_csv('./../data/wta.csv')


In [120]:
def get_period(time_diff):
    if time_diff < dt.timedelta(days=30):
        return 'last_month'
    elif time_diff < dt.timedelta(days=365):
        return 'last_year'
    else:
        return 'all'

nn_data = []

for index, row in data_matches.iterrows():
    p1 = row['Player_1']
    p2 = row['Player_2']
    surface = row['Surface']
    
    p1_data_all = dic_players.get(f'{p1}_{surface}_all', {})
    p2_data_all = dic_players.get(f'{p2}_{surface}_all', {})
    p1_data_month = dic_players.get(f'{p1}_{surface}_last_month', {})
    p2_data_month = dic_players.get(f'{p2}_{surface}_last_month', {})
    p1_data_year = dic_players.get(f'{p1}_{surface}_last_year', {})
    p2_data_year = dic_players.get(f'{p2}_{surface}_last_year', {})
    
    
    nn_data.append({
        'Player_1': p1,
        'Player_2': p2,
        'Surface': surface,
        'Date': row['Date'],
        'Player_1_Elo_All': p1_data_all.get('Elo', 1500),
        'Player_2_Elo_All': p2_data_all.get('Elo', 1500),
        'Player_1_Elo_LastYear': p1_data_year.get('Elo', 1500),
        'Player_2_Elo_LastYear': p2_data_year.get('Elo', 1500),
        'Player_1_Elo_LastMonth': p1_data_month.get('Elo', 1500),
        'Player_2_Elo_LastMonth': p2_data_month.get('Elo', 1500),

        'Player_1_Matches_All': p1_data_all.get('Matches', 0),
        'Player_2_Matches_All': p2_data_all.get('Matches', 0),
        'Player_1_WinRate_All': p1_data_all.get('WinRate', 0.5),
        'Player_2_WinRate_All': p2_data_all.get('WinRate', 0.5),

        'Player_1_WinRate_LastYear': p1_data_year.get('WinRate', 0.5),
        'Player_2_WinRate_LastYear': p2_data_year.get('WinRate', 0.5),
        'Player_1_WinRate_LastYear': p1_data_year.get('WinRate', 0.5),
        'Player_2_WinRate_LastYear': p2_data_year.get('WinRate', 0.5),
        
        'Player_1_WinRate_LastMonth': p1_data_month.get('WinRate', 0.5),
        'Player_2_WinRate_LastMonth': p2_data_month.get('WinRate', 0.5),
        'Player_1_Matches_LastYear': p1_data_year.get('Matches', 0),
        'Player_2_Matches_LastYear': p2_data_year.get('Matches', 0),
        
        'Winner': row['WinnerBinary'],
    })
    
data_nn = pd.DataFrame(nn_data)
data_nn = data_nn.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(tb.tabulate(data_nn.head(), headers='keys', tablefmt='psql'))

+----+---------------+-------------+-----------+---------------------+--------------------+--------------------+-------------------------+-------------------------+--------------------------+--------------------------+------------------------+------------------------+------------------------+------------------------+-----------------------------+-----------------------------+------------------------------+------------------------------+-----------------------------+-----------------------------+----------+
|    | Player_1      | Player_2    |   Surface | Date                |   Player_1_Elo_All |   Player_2_Elo_All |   Player_1_Elo_LastYear |   Player_2_Elo_LastYear |   Player_1_Elo_LastMonth |   Player_2_Elo_LastMonth |   Player_1_Matches_All |   Player_2_Matches_All |   Player_1_WinRate_All |   Player_2_WinRate_All |   Player_1_WinRate_LastYear |   Player_2_WinRate_LastYear |   Player_1_WinRate_LastMonth |   Player_2_WinRate_LastMonth |   Player_1_Matches_LastYear |   Player_2_Matche

In [121]:
# train, test
data_train = data_nn[data_nn['Date'] < CURRENT_DATE].reset_index(drop=True)
data_test = data_nn[data_nn['Date'] >= CURRENT_DATE].reset_index(drop=True)

X_train, y_train = data_train.drop(columns=['Date', 'Winner', 'Surface', 'Player_1', 'Player_2']), data_train['Winner']
X_test, y_test = data_test.drop(columns=['Date', 'Winner', 'Surface', 'Player_1', 'Player_2']), data_test['Winner']

In [122]:
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training target distribution: {y_train.value_counts()}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Train the model
print("Training the model...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

# Make predictions
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate additional metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Training data shape: (42513, 16)
Test data shape: (95, 16)
Training target distribution: Winner
1    21257
0    21256
Name: count, dtype: int64
Training the model...
Epoch 1/100


c:\Personal\TennisPredictor\.venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1063/1063 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6307 - loss: 0.6379 - val_accuracy: 0.6702 - val_loss: 0.6004
Epoch 2/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6307 - loss: 0.6379 - val_accuracy: 0.6702 - val_loss: 0.6004
Epoch 2/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6643 - loss: 0.6102 - val_accuracy: 0.6686 - val_loss: 0.5987
Epoch 3/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6643 - loss: 0.6102 - val_accuracy: 0.6686 - val_loss: 0.5987
Epoch 3/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6687 - loss: 0.6026 - val_accuracy: 0.6718 - val_loss: 0.5959
Epoch 4/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6687 - loss: 0.6026 - val_accuracy: 0.6718 - val_loss: 0.5959
Epoch 4/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6721 - loss: 0.6027 - val_accuracy: 0.6669 - val_loss: 0.5933
Epoch 5/100
1063/1063 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6721 - loss: 0.6027 - val_